# E-Commerce Sales & Customer Intelligence Analytics Platform
## Phase 4: Exploratory Data Analysis (EDA) & Customer Intelligence

**Author**: Senior Data Analyst & Full-Stack Developer  
**Dataset**: Brazilian E-Commerce Public Dataset by Olist (100k Orders, 2016–2018)  
**Database**: Local MySQL 8.0 (`olist_ecommerce`)  
**Currency Standard**: Brazilian Reais (R$ / BRL)  

---
### Analytical Agenda:
1. **RFM Customer Segmentation**: Quantile-discrete hybrid scoring for ~97% single-order baseline.
2. **Monthly Cohort Retention**: Empirical cohort tracking heatmap over 20+ months.
3. **Customer Lifetime Value (CLV)**: Historical, margin-adjusted, and repeat-buyer unit economics.
4. **Product Pareto Analysis**: 80/20 catalog volume concentration.
5. **Delivery SLA Breach vs. CSAT**: Linear regression of delay penalty on review scores.
6. **Macro Category Profitability**: Benchmark margin contribution across 11 sectors.
7. **Time-Series Seasonality & Forecasting**: 7-day cyclical decomposition & 30-day Holt-Winters baseline.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Database engine
sys.path.append('..')
from src.database.db_connection import get_engine

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
engine = get_engine(include_database=True)
print('Database connected successfully!')

## 1. RFM Customer Segmentation
> **Methodology Note**: In the Olist dataset, **97.0% of customers purchase only once**. To prevent quantile binning collapse on Frequency, we score Frequency discretely (1 for one-time, 2 for repeat >= 2), and segment Recency and Monetary into quartiles (1 to 4).

In [ ]:
query_rfm = '''
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    SUM(i.price) AS order_revenue
FROM dim_orders o
JOIN dim_customers c ON o.customer_id = c.customer_id
JOIN fact_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp;
'''
df_orders = pd.read_sql(query_rfm, engine)
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
ref_date = df_orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df_orders.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (ref_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('order_revenue', 'sum')
).reset_index()

rfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4]).astype(int)
rfm['f_score'] = rfm['frequency'].apply(lambda x: 2 if x >= 2 else 1)

def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if (f == 2 and r >= 3) or (r == 4 and m == 4):
        return 'Champions'
    elif (f == 2 and r <= 2) or (r >= 3 and m >= 3):
        return 'Loyal'
    elif (r == 2) or (r == 1 and m >= 3):
        return 'At Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(assign_segment, axis=1)
rfm_summary = rfm.groupby('segment').agg(
    customer_count=('customer_unique_id', 'count'),
    total_revenue_brl=('monetary', 'sum'),
    avg_recency_days=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_spend_brl=('monetary', 'mean')
).reset_index()
rfm_summary['customer_pct'] = rfm_summary['customer_count'] / len(rfm) * 100
rfm_summary['revenue_pct'] = rfm_summary['total_revenue_brl'] / rfm['monetary'].sum() * 100
rfm_summary.sort_values(by='total_revenue_brl', ascending=False)

## 2. Monthly Cohort Retention Heatmap
> Cohort retention matrix tracking active customer repurchases over subsequent months elapsed.

In [ ]:
df_orders['order_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')
cohorts = df_orders.groupby('customer_unique_id')['order_month'].min().reset_index()
cohorts.rename(columns={'order_month': 'cohort_month'}, inplace=True)
df_cohort = df_orders.merge(cohorts, on='customer_unique_id')

df_cohort['cohort_index'] = (df_cohort['order_month'].dt.year - df_cohort['cohort_month'].dt.year) * 12 + \
                             (df_cohort['order_month'].dt.month - df_cohort['cohort_month'].dt.month)
cohort_pivot = df_cohort.pivot_table(index='cohort_month', columns='cohort_index', values='customer_unique_id', aggfunc='nunique')
retention_matrix = cohort_pivot.divide(cohort_pivot[0], axis=0) * 100

plt.figure(figsize=(14, 6))
sns.heatmap(retention_matrix.loc['2017-01':'2017-12', :12], annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1.0)
plt.title('Monthly Cohort Retention Rate (%) [2017 Cohorts]', weight='bold')
plt.xlabel('Cohort Index (Months Elapsed)')
plt.ylabel('Acquisition Cohort Month')
plt.show()

## 3. Product Portfolio Pareto (80/20 Rule)

In [ ]:
query_pareto = '''
SELECT 
    i.product_id,
    p.product_category_name_english,
    COUNT(i.order_item_id) AS units_sold,
    SUM(i.price) AS total_revenue_brl
FROM fact_order_items i
JOIN dim_orders o ON i.order_id = o.order_id
JOIN dim_products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
GROUP BY i.product_id, p.product_category_name_english
ORDER BY total_revenue_brl DESC;
'''
df_pareto = pd.read_sql(query_pareto, engine)
total_prod_rev = df_pareto['total_revenue_brl'].sum()
df_pareto['cum_rev_pct'] = df_pareto['total_revenue_brl'].cumsum() / total_prod_rev * 100
df_pareto['cum_prod_pct'] = (np.arange(1, len(df_pareto) + 1)) / len(df_pareto) * 100

cutoff_80 = len(df_pareto[df_pareto['cum_rev_pct'] <= 80]) + 1
print(f'Products generating 80% GMV: {cutoff_80:,} of {len(df_pareto):,} ({cutoff_80/len(df_pareto)*100:.2f}%)')

plt.figure(figsize=(10, 5))
plt.plot(df_pareto['cum_prod_pct'], df_pareto['cum_rev_pct'], color='#2980b9', lw=2.5)
plt.axhline(80, color='#e74c3c', linestyle='--')
plt.axvline(cutoff_80/len(df_pareto)*100, color='#27ae60', linestyle='--')
plt.title('Product Portfolio Pareto Curve (80/20)', weight='bold')
plt.xlabel('Cumulative % of Products')
plt.ylabel('Cumulative % of Revenue')
plt.show()

## 4. Delivery Delay vs CSAT Review Score

In [ ]:
query_csat = '''
SELECT 
    o.order_id,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score
FROM dim_orders o
JOIN fact_order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;
'''
df_csat = pd.read_sql(query_csat, engine)
df_csat['delivered_date'] = pd.to_datetime(df_csat['order_delivered_customer_date']).dt.date
df_csat['estimated_date'] = pd.to_datetime(df_csat['order_estimated_delivery_date']).dt.date
df_csat['delay_days'] = (df_csat['delivered_date'] - df_csat['estimated_date']).apply(lambda x: x.days)

late_csat = df_csat[df_csat['delay_days'] > 0]
slope, intercept, r_val, p_val, std_err = stats.linregress(late_csat['delay_days'], late_csat['review_score'])
print(f'Linear regression slope: {slope:.4f} stars/day (p={p_val:.2e}, R^2={r_val**2:.4f})')

def bucket_delay(d):
    if d <= 0: return '0. On-Time (<= 0d)'
    elif d <= 3: return '1. Minor (1-3d)'
    elif d <= 7: return '2. Moderate (4-7d)'
    elif d <= 14: return '3. High (8-14d)'
    else: return '4. Severe (15+d)'

df_csat['bucket'] = df_csat['delay_days'].apply(bucket_delay)
csat_summary = df_csat.groupby('bucket')['review_score'].agg(['count', 'mean']).reset_index()
csat_summary

## 5. Seasonality Decomposition & 30-Day Holt-Winters Baseline Forecast

In [ ]:
query_daily = '''
SELECT 
    DATE(o.order_purchase_timestamp) AS order_date,
    SUM(i.price) AS daily_revenue
FROM dim_orders o
JOIN fact_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY DATE(o.order_purchase_timestamp)
ORDER BY order_date;
'''
df_daily = pd.read_sql(query_daily, engine)
df_daily['order_date'] = pd.to_datetime(df_daily['order_date'])
df_daily.set_index('order_date', inplace=True)
df_daily = df_daily.asfreq('D', fill_value=0)

# Train/Test split: 30-day holdout
train = df_daily['daily_revenue'].iloc[:-30]
test = df_daily['daily_revenue'].iloc[-30:]

hw = ExponentialSmoothing(train, seasonal_periods=7, trend='add', seasonal='add', initialization_method='estimated').fit()
pred = hw.forecast(30)

wape = (np.sum(np.abs(test - pred)) / np.sum(test)) * 100
mae = np.mean(np.abs(test - pred))
print(f'Test Mean: R$ {test.mean():,.2f} | Forecast Mean: R$ {pred.mean():,.2f}')
print(f'Validation MAE: R$ {mae:,.2f} | WAPE: {wape:.2f}%')

plt.figure(figsize=(12, 5))
plt.plot(train.index[-60:], train.iloc[-60:], label='Train (Last 60 Days)', color='#7f8c8d')
plt.plot(test.index, test, label='Actual Test (Aug 2018)', color='#2980b9', lw=2)
plt.plot(test.index, pred, label='Holt-Winters Forecast', color='#e67e22', lw=2, linestyle='--')
plt.title('Holt-Winters 30-Day Holdout Forecast Evaluation', weight='bold')
plt.legend()
plt.show()